# 97. The Last-Mile Delivery & Micro-Fulfillment Problem
## Tier 2: The Classic Heuristic (Sweep Algorithm Implementation)

### Key assumptions
- Customers are distributed around a central depot location
- Vehicle capacity constraints must be respected
- Route efficiency is measured by total distance traveled
- Geographic clustering leads to efficient routes
- Angular positioning provides natural route segmentation

### Approach (step-by-step)
1. **Polar Coordinate Conversion**: Convert customer locations to polar angles relative to depot
2. **Angular Sorting**: Sort customers by angle from starting direction (positive x-axis)
3. **Sequential Assignment**: Sweep through customers, assigning to current route until capacity limit
4. **Route Creation**: Start new route when capacity would be exceeded
5. **Distance Calculation**: Compute total distance for each generated route

### What to look for in the results
- Geographically cohesive customer clusters
- Vehicle capacity utilization patterns
- Total distance efficiency compared to optimal solution
- Number of routes required vs available vehicles

### Concrete example (from the source)
Implementation of sweep algorithm with depot at (50, 50) serving 8 customers, demonstrating the angular clustering approach.

In [1]:
# Import required libraries for sweep algorithm implementation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict
import math
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class Customer:
    """Represents a customer location with demand"""
    id: int
    x: float
    y: float
    demand: int
    angle: float = 0.0  # Polar angle relative to depot
    distance_from_depot: float = 0.0

@dataclass
class Route:
    """Represents a delivery route with assigned customers"""
    id: int
    customers: List[Customer]
    total_demand: int = 0
    total_distance: float = 0.0
    
    def add_customer(self, customer: Customer):
        """Add customer to route and update totals"""
        self.customers.append(customer)
        self.total_demand += customer.demand

def calculate_euclidean_distance(x1: float, y1: float, x2: float, y2: float) -> float:
    """Calculate Euclidean distance between two points"""
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

def calculate_polar_angle(depot_x: float, depot_y: float, customer_x: float, customer_y: float) -> float:
    """Calculate polar angle of customer relative to depot (in radians)"""
    dx = customer_x - depot_x
    dy = customer_y - depot_y
    angle = math.atan2(dy, dx)
    # Convert negative angles to positive (0 to 2π)
    if angle < 0:
        angle += 2 * math.pi
    return angle

print("Sweep algorithm data structures and utilities defined")

In [ ]:
# Initialize the concrete example from the problem statement
# Depot at (50, 50) with 8 customers
depot_location = (50, 50)
vehicle_capacity = 10  # Capacity per vehicle

# Customer data from the example
customers_data = [
    (0, 65, 45, 3),   # Customer 0: (65, 45), demand 3
    (1, 70, 60, 2),   # Customer 1: (70, 60), demand 2
    (2, 30, 70, 4),   # Customer 2: (30, 70), demand 4
    (3, 25, 55, 3),   # Customer 3: (25, 55), demand 3
    (4, 40, 30, 2),   # Customer 4: (40, 30), demand 2
    (5, 60, 25, 1),   # Customer 5: (60, 25), demand 1
    (6, 75, 35, 3),   # Customer 6: (75, 35), demand 3
    (7, 20, 40, 2),   # Customer 7: (20, 40), demand 2
]

# Create Customer objects
customers = []
for cust_id, x, y, demand in customers_data:
    angle = calculate_polar_angle(depot_location[0], depot_location[1], x, y)
    distance = calculate_euclidean_distance(depot_location[0], depot_location[1], x, y)
    customers.append(Customer(cust_id, x, y, demand, angle, distance))

print(f"Initialized {len(customers)} customers around depot at {depot_location}")
print(f"Vehicle capacity: {vehicle_capacity} units")

# Display customers with polar coordinates
print("\nCustomer polar coordinates:")
for customer in customers:
    print(f"Customer {customer.id}: ({customer.x}, {customer.y}), demand={customer.demand}, "
          f"angle={math.degrees(customer.angle):.1f}°, distance={customer.distance_from_depot:.1f}")

In [ ]:
# Sort customers by polar angle (counterclockwise from positive x-axis)
customers_sorted = sorted(customers, key=lambda c: c.angle)

print("Customers sorted by polar angle:")
for i, customer in enumerate(customers_sorted):
    print(f"{i}: Customer {customer.id}, angle={math.degrees(customer.angle):.1f}°, demand={customer.demand}")

# Implement sweep algorithm
def sweep_algorithm(customers: List[Customer], depot: Tuple[float, float], 
                   vehicle_capacity: int) -> List[Route]:
    """Implement sweep algorithm for vehicle routing"""
    
    # Sort customers by polar angle
    customers_sorted = sorted(customers, key=lambda c: c.angle)
    
    routes = []
    current_route = None
    route_id = 1
    
    for customer in customers_sorted:
        # Start new route if needed
        if current_route is None:
            current_route = Route(route_id, [])
            route_id += 1
        
        # Check if adding customer would exceed capacity
        if current_route.total_demand + customer.demand <= vehicle_capacity:
            # Add customer to current route
            current_route.add_customer(customer)
        else:
            # Finish current route and start new one
            routes.append(current_route)
            current_route = Route(route_id, [])
            route_id += 1
            current_route.add_customer(customer)
    
    # Add the last route if it has customers
    if current_route and current_route.customers:
        routes.append(current_route)
    
    # Calculate distances for each route
    for route in routes:
        route.total_distance = calculate_route_distance(route, depot)
    
    return routes

def calculate_route_distance(route: Route, depot: Tuple[float, float]) -> float:
    """Calculate total distance for a route (depot -> customers -> depot)"""
    if not route.customers:
        return 0.0
    
    total_distance = 0.0
    current_x, current_y = depot
    
    # Distance from depot to first customer
    first_customer = route.customers[0]
    total_distance += calculate_euclidean_distance(current_x, current_y, 
                                                   first_customer.x, first_customer.y)
    
    # Distance between consecutive customers
    for i in range(len(route.customers) - 1):
        curr_customer = route.customers[i]
        next_customer = route.customers[i + 1]
        total_distance += calculate_euclidean_distance(curr_customer.x, curr_customer.y,
                                                       next_customer.x, next_customer.y)
    
    # Distance from last customer back to depot
    last_customer = route.customers[-1]
    total_distance += calculate_euclidean_distance(last_customer.x, last_customer.y,
                                                   depot[0], depot[1])
    
    return total_distance

print("Sweep algorithm implementation complete")

In [ ]:
# Run sweep algorithm
routes = sweep_algorithm(customers, depot_location, vehicle_capacity)

print("=" * 60)
print("SWEEP ALGORITHM RESULTS")
print("=" * 60)

# Display results
total_distance = 0
total_demand = 0
vehicle_utilization = []

for route in routes:
    utilization = (route.total_demand / vehicle_capacity) * 100
    vehicle_utilization.append(utilization)
    total_distance += route.total_distance
    total_demand += route.total_demand
    
    print(f"\n🚚 Route {route.id}:")
    print(f"   Customers: {[c.id for c in route.customers]}")
    print(f"   Total demand: {route.total_demand} units")
    print(f"   Capacity utilization: {utilization:.1f}%")
    print(f"   Total distance: {route.total_distance:.2f} units")
    
    # Show customer coordinates for this route
    print(f"   Customer coordinates: {[(c.x, c.y) for c in route.customers]}")

print(f"\n📊 SUMMARY:")
print(f"   Total routes created: {len(routes)}")
print(f"   Total distance: {total_distance:.2f} units")
print(f"   Total demand served: {total_demand} units")
print(f"   Average vehicle utilization: {np.mean(vehicle_utilization):.1f}%")
print(f"   Average route efficiency: {total_distance/total_demand:.2f} distance per demand unit")

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Sweep Algorithm for Last-Mile Delivery', fontsize=16, fontweight='bold')

# 1. Geographic visualization with routes
ax1 = axes[0, 0]
route_colors = ['red', 'blue', 'green', 'orange', 'purple']

# Plot depot
ax1.plot(depot_location[0], depot_location[1], 'D', markersize=20, 
        color='black', label='Depot', zorder=5)

# Plot sweep direction indicator
sweep_line_length = 30
ax1.arrow(depot_location[0], depot_location[1], sweep_line_length, 0, 
          head_width=3, head_length=3, fc='gray', ec='gray', alpha=0.5, 
          label='Sweep start direction')

# Plot routes
for i, route in enumerate(routes):
    color = route_colors[i % len(route_colors)]
    
    # Plot customers
    for customer in route.customers:
        ax1.plot(customer.x, customer.y, 'o', markersize=10, color=color, 
                alpha=0.7, label=f'Route {route.id}' if customer == route.customers[0] else '')
        ax1.text(customer.x + 1, customer.y + 1, str(customer.id), fontsize=9)
    
    # Draw route path
    route_points = [(depot_location[0], depot_location[1])] + \
                   [(c.x, c.y) for c in route.customers] + \
                   [(depot_location[0], depot_location[1])]
    
    for j in range(len(route_points) - 1):
        ax1.plot([route_points[j][0], route_points[j+1][0]], 
                [route_points[j][1], route_points[j+1][1]], 
                color=color, alpha=0.6, linewidth=2)

ax1.set_xlabel('X Coordinate')
ax1.set_ylabel('Y Coordinate')
ax1.set_title('Geographic Distribution and Sweep Routes')
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.set_xlim(10, 80)
ax1.set_ylim(20, 80)

# 2. Polar angle distribution
ax2 = axes[0, 1]
angles_degrees = [math.degrees(c.angle) for c in customers_sorted]
demands = [c.demand for c in customers_sorted]
customer_ids = [c.id for c in customers_sorted]

# Create bar plot of customers by angle
bars = ax2.bar(range(len(customers_sorted)), demands, alpha=0.7, 
               color=[route_colors[get_route_for_customer(c.id, routes)-1] for c in customers_sorted])

ax2.set_xlabel('Customer Order (by Polar Angle)')
ax2.set_ylabel('Demand')
ax2.set_title('Customer Assignment by Polar Angle')
ax2.set_xticks(range(len(customers_sorted)))
ax2.set_xticklabels([f'C{id}\n{math.degrees(c.angle):.0f}°' for c, id in zip(customers_sorted, customer_ids)], 
                   rotation=45)

# Add route boundaries
current_route_id = 0
for i, customer in enumerate(customers_sorted):
    route_id = get_route_for_customer(customer.id, routes)
    if route_id != current_route_id:
        ax2.axvline(x=i-0.5, color='gray', linestyle='--', alpha=0.5)
        current_route_id = route_id

# 3. Vehicle capacity utilization
ax3 = axes[1, 0]
route_names = [f'Route {i+1}' for i in range(len(routes))]
utilizations = [(r.total_demand / vehicle_capacity) * 100 for r in routes]

bars = ax3.bar(route_names, utilizations, color=route_colors[:len(routes)], alpha=0.7)
ax3.set_xlabel('Route')
ax3.set_ylabel('Capacity Utilization (%)')
ax3.set_title('Vehicle Capacity Utilization')
ax3.set_ylim(0, 100)

# Add percentage labels on bars
for bar, util in zip(bars, utilizations):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'{util:.1f}%', ha='center', va='bottom', fontweight='bold')

# 4. Route distance efficiency
ax4 = axes[1, 1]
distances = [r.total_distance for r in routes]
demands_per_route = [r.total_demand for r in routes]
efficiency = [d/d if d > 0 else 0 for d in demands_per_route]  # Distance per demand unit

bars = ax4.bar(route_names, distances, color=route_colors[:len(routes)], alpha=0.7)
ax4.set_xlabel('Route')
ax4.set_ylabel('Total Distance')
ax4.set_title('Route Distance Comparison')

# Add distance labels on bars
for bar, dist, demand in zip(bars, distances, demands_per_route):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{dist:.1f}\n({demand} units)', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("Sweep algorithm visualization complete!")

In [ ]:
# Helper function to get route for customer
def get_route_for_customer(customer_id: int, routes: List[Route]) -> int:
    """Find which route contains a specific customer"""
    for route in routes:
        if any(c.id == customer_id for c in route.customers):
            return route.id
    return 0

# Performance analysis
print("=" * 60)
print("PERFORMANCE ANALYSIS")
print("=" * 60)

# Calculate efficiency metrics
total_customers = len(customers)
total_demand_served = sum(r.total_demand for r in routes)
total_distance_all_routes = sum(r.total_distance for r in routes)
avg_distance_per_customer = total_distance_all_routes / total_customers
avg_distance_per_demand = total_distance_all_routes / total_demand_served

print(f"📈 EFFICIENCY METRICS:")
print(f"   Customers served: {total_customers}")
print(f"   Total demand served: {total_demand_served} units")
print(f"   Total distance traveled: {total_distance_all_routes:.2f} units")
print(f"   Average distance per customer: {avg_distance_per_customer:.2f} units")
print(f"   Average distance per demand unit: {avg_distance_per_demand:.2f} units")
print(f"   Vehicle utilization: {np.mean(vehicle_utilization):.1f}%")

# Theoretical comparison
print(f"\n🔍 ALGORITHM CHARACTERISTICS:")
print(f"   Time complexity: O(n log n) due to sorting")
print(f"   Space complexity: O(n) for storing customers")
print(f"   Solution quality: Good for geographically clustered customers")
print(f"   Scalability: Excellent for large problem instances")

# Comparison with expected results from problem statement
print(f"\n📋 COMPARISON WITH EXPECTED RESULTS:")
print(f"   Expected routes: 3")
print(f"   Actual routes: {len(routes)}")
print(f"   Expected utilization: ~95%")
print(f"   Actual utilization: {np.mean(vehicle_utilization):.1f}%")
print(f"   Expected efficiency: ~1.23 distance per demand")
print(f"   Actual efficiency: {avg_distance_per_demand:.2f} distance per demand")

### Why this Tier exists vs Tier 1
The Sweep Algorithm provides practical advantages over the mathematical formulation:
- **Computational efficiency**: O(n log n) vs exponential complexity of MIP
- **Scalability**: Handles thousands of customers vs limited by MIP solver
- **Real-time performance**: Suitable for dynamic routing vs offline optimization
- **Geographic intuition**: Natural clustering based on location vs abstract mathematical assignment

### Pros / Cons vs Tier 1 (Mathematical Formulation)
**Pros:**
- Much faster execution time (milliseconds vs minutes/hours)
- Handles large problem instances easily
- Produces geographically coherent routes
- Simple to implement and understand
- Suitable for real-time applications

**Cons:**
- No optimality guarantee (heuristic solution)
- May miss better non-geographic route combinations
- Sensitive to depot location choice
- Doesn't consider time windows or complex constraints
- Performance depends on customer distribution pattern

### When to use this Tier
- Large-scale delivery networks (100+ customers)
- Real-time routing requirements
- When geographic clustering is natural
- Limited computational resources
- As initial solution for more sophisticated algorithms
- When speed is more important than optimality